# Resumable validation of trained VQCs on IBM Quantum hardware

This notebook was adapted to the structure of the pickle files produced by the training pipeline.

Workflow:

1. Locate the experiment pickle files.
2. For each **disorder + ansatz**, select the repetition with the highest `f1_weighted`.
3. Retrieve exactly the weights saved for that repetition.
4. Reconstruct the same `train/test split` using the saved `random_state`.
5. Reapply PCA/UMAP and the quantum scaler using only the training data.
6. Verify that the reconstructed `y_test` is identical to the saved `y_true`.
7. Run the trained weights on a **fixed IBM QPU**, without automatically switching processors.
8. Compare:
   - the saved simulator result;
   - the real IBM Quantum result;
   - the best saved classical model on the same test set.
9. Store backend metadata, calibration properties, jobs, counts, transpilation data, metrics, and QASM files.

> The IBM token is not stored in the results.

This notebook uses persistent checkpoints and can recover previously submitted IBM jobs.


In [ ]:
# Instale the single time, if required:
# %pip install -U "qiskit>=2.0" "qiskit-ibm-runtime>=0.40" numpy pandas scikit-learn umap-learn


## 1. Imports and helper functions

The cell below contains the same functions as the `.py` version, without `argparse`/CLI.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
validate_vqc_ibm_hardware.py

validates, in hardware real of the IBM Quantum, the VQCs already trained and saved
in the pickles produced by the pipeline of health mental.

principle of selection
--------------------
by default, for each (disorder, ansatz), the script percorre all the
pickles/configurations/repetitions and seleciona the repetition with the highest
f1_weighted. the weights ("params") used in the hardware are exactly the
saved nessa repetition.

For the classical comparison, the script searches, for the same disorder and the
same outer test split (same random_state and y_true), the model classical
with highest f1_weighted between the results saved in the pickles.

important
----------
The training pickle contains:
  - weights of the VQCs;
  - configuration;
  - random_state;
  - y_true/y_pred;
  - metrics;
  - models/pipelines classical.

However, it does NOT contain the already preprocessed X_test, nor the PCA/UMAP and quantum scaler
already fitted. Therefore, to reproduce exactly the vectors sent
to the QPU, this script also requires of the CSV original used in the training.
It reconstructs the same split and the same preprocessing, with the same seed, and
ABORTA if the y_test reconstructed not for identical to the y_true saved in the pickle.

the token of the IBM not is gravado in the files of output.

Dependencies
------------
pip install "qiskit>=2.0" "qiskit-ibm-runtime>=0.40" \
            numpy pandas scikit-learn umap-learn

example
-------
export IBM_QUANTUM_TOKEN="SEU_TOKEN"
export IBM_QUANTUM_INSTANCE="SEU_CRN_DA_INSTANCIA"

python validate_vqc_ibm_hardware.py \
    --pickles "results/*.pkl" \
    --csv dataset_final_mental_health.csv \
    --backend ibm_fez \
    --shots 8192 \
    --output ibm_validation

for listar backends reais available:
python validate_vqc_ibm_hardware.py --list-backends

for only verify quais models/repeats seriam escolhidos:
python validate_vqc_ibm_hardware.py \
    --pickles "results/*.pkl" \
    --csv dataset_final_mental_health.csv \
    --backend ibm_fez \
    --dry-run
"""

from __future__ import annotations

import argparse
import glob
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import pickle
import platform
import socket
import sys
import time
import traceback
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd

from collections import defaultdict

from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler


QUANTUM_MODELS = {
    "hardware_efficient",
    "strongly_entangling",
    "basic_entangler",
    "qaoa_inspired",
    "tree_tensor",
}

CLASSICAL_MODELS = {
    "svm_linear",
    "svm_rbf",
    "svm_poly",
    "svm_sigmoid",
    "decision_tree",
    "mlp",
    "random_forest",
}


# =============================================================================
# Utilidades
# =============================================================================

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def jsonable(obj: Any):
    if obj is None or isinstance(obj, (str, int, bool)):
        return obj

    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return str(obj)
        return obj

    if isinstance(obj, np.generic):
        return jsonable(obj.item())

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, datetime):
        return obj.isoformat()

    if isinstance(obj, Mapping):
        return {str(k): jsonable(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [jsonable(v) for v in obj]

    if hasattr(obj, "to_dict"):
        try:
            return jsonable(obj.to_dict())
        except Exception:
            pass

    return repr(obj)


def save_json(path: Path, obj: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(jsonable(obj), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def safe_name(text: str) -> str:
    chars = []
    for c in str(text):
        chars.append(c if c.isalnum() or c in "-_." else "_")
    return "".join(chars)


def sha256_file(path: str | Path, block_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def package_versions():
    packages = [
        "qiskit",
        "qiskit-ibm-runtime",
        "numpy",
        "pandas",
        "scikit-learn",
        "umap-learn",
    ]
    result = {}
    for pkg in packages:
        try:
            result[pkg] = importlib_metadata.version(pkg)
        except Exception:
            result[pkg] = None
    return result


def parse_config_key(config_key: str) -> dict[str, Any]:
    """Converte the key textual of the seu pipeline in dictionary."""
    out: dict[str, Any] = {}
    for part in str(config_key).split("|"):
        if "=" not in part:
            continue
        key, value = part.split("=", 1)
        lv = value.lower()

        if lv == "true":
            parsed = True
        elif lv == "false":
            parsed = False
        elif lv in {"none", "null"}:
            parsed = None
        else:
            try:
                parsed = int(value)
            except ValueError:
                try:
                    parsed = float(value)
                except ValueError:
                    parsed = value

        out[key] = parsed
    return out


# =============================================================================
# Preprocessing: replica of the functions used during training
# =============================================================================

def prepare_disorder_data(
    df,
    disorder_main,
    column_main="main.disorder",
    column_specific="specific.disorder",
    group_control="Healthy control",
    columns_exclude=("sex_M",),
    remove_rows_with_nulls=True,
):
    """
    Reproduz the function preparar_dados_desordem() of the pipeline of training.
    """
    mask = (
        (df[column_main] == disorder_main)
        | (df[column_main] == group_control)
    )
    df_sub = df[mask].copy()

    cols_remove = [column_main, column_specific, *columns_exclude]
    cols_remove = [c for c in cols_remove if c in df_sub.columns]

    X_df = df_sub.drop(columns=cols_remove)
    y_raw = df_sub[column_specific]

    non_numeric = X_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        X_df = X_df.drop(columns=non_numeric)

    if remove_rows_with_nulls and X_df.isnull().values.any():
        valid = ~X_df.isnull().any(axis=1)
        X_df = X_df[valid]
        y_raw = y_raw[valid]

    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    return (
        X_df.to_numpy(dtype=float),
        np.asarray(y),
        list(le.classes_),
    )


def create_reducer(metodo="pca", n_components=4, random_state=42):
    if metodo == "pca":
        return PCA(
            n_components=n_components,
            random_state=random_state,
            svd_solver="full",
        )

    if metodo == "umap":
        try:
            import umap
        except ImportError as exc:
            raise ImportError(
                "UMAP requerido. Instale with: pip install umap-learn"
            ) from exc

        return umap.UMAP(
            n_components=n_components,
            random_state=random_state,
            n_jobs=1,
        )

    raise ValueError("metodo should be 'pca' or 'umap'")


def preprocess_quantum_split(
    X_train,
    X_test,
    embedding_type="angle",
    dim_reduction=None,
    n_final_features=None,
    max_qubits=6,
    random_state=42,
):
    """
    Reproduz the function preprocessar_split_quantico() of the seu training.
    """
    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)

    reducer = None

    if dim_reduction is not None:
        if n_final_features is None:
            raise ValueError("n_final_features is required when using PCA/UMAP.")

        reducer = create_reducer(
            dim_reduction,
            n_components=int(n_final_features),
            random_state=random_state,
        )
        X_train_proc = reducer.fit_transform(X_train)
        X_test_proc = reducer.transform(X_test)

    elif embedding_type == "angle" and X_train.shape[1] > int(max_qubits):
        reducer = create_reducer(
            "pca",
            n_components=int(max_qubits),
            random_state=random_state,
        )
        X_train_proc = reducer.fit_transform(X_train)
        X_test_proc = reducer.transform(X_test)

    else:
        X_train_proc = X_train.copy()
        X_test_proc = X_test.copy()

    if embedding_type == "angle":
        scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)
    elif embedding_type == "amplitude":
        scaler = StandardScaler()
    else:
        raise ValueError("embedding_type should be 'angle' or 'amplitude'.")

    X_train_proc = scaler.fit_transform(X_train_proc)
    X_test_proc = scaler.transform(X_test_proc)

    return X_train_proc, X_test_proc, reducer, scaler


# =============================================================================
# readout of the pickles
# =============================================================================

def resolve_pickle_paths(patterns: Sequence[str]) -> list[Path]:
    paths: list[Path] = []

    for pattern in patterns:
        matches = [Path(p) for p in glob.glob(pattern)]
        if not matches and Path(pattern).exists():
            matches = [Path(pattern)]
        paths.extend(matches)

    unique = []
    seen = set()

    for p in paths:
        rp = p.resolve()
        if rp not in seen and rp.is_file() and rp.suffix.lower() == ".pkl":
            unique.append(rp)
            seen.add(rp)

    if not unique:
        raise FileNotFoundError(
            "in the pickle file found. Example: --pickles 'results/*.pkl'"
        )

    return sorted(unique)


def load_pickle(path: Path):
    with path.open("rb") as f:
        return pickle.load(f)


def iter_records(pickle_paths: Sequence[Path]):
    """
    Extrai the row conceitual by (configuration, disorder, model, repetition)
    directly from the structure produced by the user-provided pipeline.
    """
    for pkl_path in pickle_paths:
        data = load_pickle(pkl_path)
        pkl_hash = sha256_file(pkl_path)

        if not isinstance(data, dict):
            continue

        for config_key, config_block in data.items():
            if str(config_key).startswith("_") or not isinstance(config_block, dict):
                continue

            config_from_key = parse_config_key(config_key)
            config_metadata = config_block.get("_metadata", {})

            for disorder, disorder_block in config_block.items():
                if str(disorder).startswith("_") or not isinstance(disorder_block, dict):
                    continue

                disorder_metadata = disorder_block.get("_metadata", {})
                classes = disorder_metadata.get("class_idxs")

                for model_name, model_block in disorder_block.items():
                    if str(model_name).startswith("_") or not isinstance(model_block, dict):
                        continue

                    if "repeticoes" in model_block:
                        repetitions = model_block["repeticoes"]
                    elif "f1_weighted" in model_block:
                        repetitions = [model_block]
                    else:
                        continue

                    for repeat_index, rep in enumerate(repetitions):
                        if not isinstance(rep, dict) or "f1_weighted" not in rep:
                            continue

                        rep_config = rep.get("config", {}) or {}

                        merged_config = {}
                        merged_config.update(config_from_key)
                        if isinstance(config_metadata, dict):
                            merged_config.update(config_metadata)
                        if isinstance(disorder_metadata, dict):
                            merged_config.update(disorder_metadata)
                        merged_config.update(rep_config)

                        random_state = rep_config.get("random_state")
                        if random_state is None:
                            dataset_seed = merged_config.get("random_state_base", 42)
                            random_state = int(dataset_seed) + int(repeat_index)

                        yield {
                            "pickle_path": str(pkl_path),
                            "pickle_sha256": pkl_hash,
                            "config_key": str(config_key),
                            "config_from_key": config_from_key,
                            "config_metadata": config_metadata,
                            "disorder_metadata": disorder_metadata,
                            "merged_config": merged_config,
                            "disorder": str(disorder),
                            "class_idxs": classes,
                            "model_name": str(model_name),
                            "repeat_index": int(repeat_index),
                            "random_state": int(random_state),
                            "f1_weighted": float(rep["f1_weighted"]),
                            "metrics": rep,
                        }


def choose_quantum_records(records, selection="per-disorder-ansatz"):
    """
    selection in two etapas.

    for each (disorder, ansatz):
      1. Agrupa the repeats of the same configuration.
      2. Compute the configuration's mean F1.
      3. Select the configuration with the highest mean F1.
      4. inside dela, seleciona the repeat of highest F1.

    Therefore:
        better configuration -> better repeat -> weights for IBM
    """

    quantum = [
        r for r in records
        if r["model_name"] in QUANTUM_MODELS
    ]

    if not quantum:
        raise RuntimeError(
            "in the VQC was found in the pickles."
        )

    # ============================================================
    # 1. Agrupa repeats of the same configuration
    # ============================================================

    configurations = defaultdict(list)

    for r in quantum:

        config_id = (
            r["disorder"],
            r["model_name"],
            r["config_key"],
        )

        configurations[config_id].append(r)

    # ============================================================
    # 2. Compute the mean F1 of each configuration
    # ============================================================

    config_candidates = []

    for config_id, repetitions in configurations.items():

        f1_values = np.asarray(
            [
                rep["f1_weighted"]
                for rep in repetitions
            ],
            dtype=float,
        )

        config_candidates.append({
            "config_id": config_id,

            "mean_f1": float(
                np.mean(f1_values)
            ),

            "std_f1": float(
                np.std(f1_values)
            ),

            "n_repeats": len(repetitions),

            "repetitions": repetitions,
        })

    # ============================================================
    # 3. Agrupa configurations by dataset + ansatz
    # ============================================================

    per_ansatz_groups = defaultdict(list)

    for cfg in config_candidates:

        disorder, ansatz, _ = cfg["config_id"]

        per_ansatz_groups[
            (disorder, ansatz)
        ].append(cfg)

    winners_per_ansatz = []

    # ============================================================
    # 4. better configuration of each ansatz
    # ============================================================

    for (disorder, ansatz), candidates in per_ansatz_groups.items():

        candidates = sorted(
            candidates,
            key=lambda cfg: (
                -cfg["mean_f1"],       # highest mean first
                cfg["std_f1"],         # desempate: menor std
                str(cfg["config_id"][2]),
            ),
        )

        best_cfg = candidates[0]

        # ========================================================
        # 5. better repeat inside of the configuration vencedora
        # ========================================================

        best_repeat = sorted(
            best_cfg["repetitions"],
            key=lambda r: (
                -r["f1_weighted"],
                r["repeat_index"],
                r["pickle_path"],
            ),
        )[0]

        best_repeat = dict(best_repeat)

        best_repeat.update({
            "configuration_mean_f1":
                best_cfg["mean_f1"],

            "configuration_std_f1":
                best_cfg["std_f1"],

            "configuration_n_repeats":
                best_cfg["n_repeats"],
        })

        winners_per_ansatz.append(best_repeat)

    winners_per_ansatz = sorted(
        winners_per_ansatz,
        key=lambda r: (
            r["disorder"],
            r["model_name"],
        ),
    )

    # ============================================================
    # the vencedor for each ansatz of each dataset
    # ============================================================

    if selection == "per-disorder-ansatz":
        return winners_per_ansatz

    # ============================================================
    # the single ansatz vencedor for each dataset
    # ============================================================

    if selection == "per-disorder":

        grouped = defaultdict(list)

        for r in winners_per_ansatz:
            grouped[r["disorder"]].append(r)

        selected = []

        for disorder, candidates in grouped.items():

            candidates = sorted(
                candidates,
                key=lambda r: (
                    -r["configuration_mean_f1"],
                    r["configuration_std_f1"],
                    r["model_name"],
                ),
            )

            selected.append(
                candidates[0]
            )

        return sorted(
            selected,
            key=lambda r: r["disorder"]
        )

    raise ValueError(selection)


def choose_best_classical_same_split(qrec, all_records):
    """
    Seleciona the classical of highest F1 in the same held-out split.

    the igualdade of y_true is verificada explicitly for impedir comparison
    acidental between splits distintos.
    """
    q_y_true = np.asarray(qrec["metrics"]["y_true"])
    candidates = []

    for rec in all_records:
        if rec["model_name"] not in CLASSICAL_MODELS:
            continue
        if rec["disorder"] != qrec["disorder"]:
            continue
        if rec["random_state"] != qrec["random_state"]:
            continue

        c_y_true = np.asarray(rec["metrics"].get("y_true", []))

        if c_y_true.shape == q_y_true.shape and np.array_equal(c_y_true, q_y_true):
            candidates.append(rec)

    if not candidates:
        return None

    return max(candidates, key=lambda r: r["f1_weighted"])


def selection_table(selected, all_records):

    rows = []

    for q in selected:

        c = choose_best_classical_same_split(
            q,
            all_records,
        )

        rows.append({

            "disorder":
                q["disorder"],

            "rank_ansatz_in_disorder":
                q.get(
                    "ansatz_rank_within_disorder"
                ),

            "quantum_model":
                q["model_name"],

            # better configuration
            "configuration_mean_f1":
                q.get(
                    "configuration_mean_f1"
                ),

            "configuration_std_f1":
                q.get(
                    "configuration_std_f1"
                ),

            "configuration_n_repeats":
                q.get(
                    "configuration_n_repeats"
                ),

            # Repeat used in the IBM
            "best_repeat_f1":
                q["f1_weighted"],

            "repeat_index":
                q["repeat_index"],

            "random_state":
                q["random_state"],

            "config_key":
                q["config_key"],

            "pickle":
                q["pickle_path"],

            "classical_model_same_split":
                None if c is None
                else c["model_name"],

            "classical_f1_same_split":
                None if c is None
                else c["f1_weighted"],
        })

    return pd.DataFrame(rows)


# =============================================================================
# Reconstruction of the held-out test set
# =============================================================================

def reconstruct_test_set(
    df: pd.DataFrame,
    record: dict,
    test_size: float,
    column_main: str,
    column_specific: str,
    group_control: str,
    columns_exclude: tuple[str, ...],
):
    cfg = record["merged_config"]
    random_state = int(record["random_state"])

    X, y, class_names = prepare_disorder_data(
        df,
        record["disorder"],
        column_main=column_main,
        column_specific=column_specific,
        group_control=group_control,
        columns_exclude=columns_exclude,
    )

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    embedding_type = cfg.get("embedding_type", "angle")
    dim_reduction = cfg.get("reducao_dim")
    n_final_features = cfg.get("n_features_final")
    max_qubits = cfg.get("n_qubits", cfg.get("n_qubits_max", n_final_features or 6))

    X_train_q, X_test_q, reducer, scaler = preprocess_quantum_split(
        X_train_raw,
        X_test_raw,
        embedding_type=embedding_type,
        dim_reduction=dim_reduction,
        n_final_features=n_final_features,
        max_qubits=max_qubits,
        random_state=random_state,
    )

    y_saved = np.asarray(record["metrics"]["y_true"])

    if y_test.shape != y_saved.shape or not np.array_equal(y_test, y_saved):
        raise RuntimeError(
            "\nFALHA of reproduction of the SPLIT.\n"
            f"Disorder: {record['disorder']}\n"
            f"Model: {record['model_name']}\n"
            f"Seed: {random_state}\n"
            "the y_test reconstructed not is identical to the y_true saved in the pickle.\n"
            "No job will be submitted for this model. Verify that the CSV, "
            "columns excluded and test_size are exactly the of the training."
        )

    return {
        "X_train_raw": X_train_raw,
        "X_test_raw": X_test_raw,
        "X_train_q": np.asarray(X_train_q, dtype=float),
        "X_test_q": np.asarray(X_test_q, dtype=float),
        "y_train": np.asarray(y_train),
        "y_test": np.asarray(y_test),
        "class_names": class_names,
        "reducer": reducer,
        "scaler": scaler,
    }


# =============================================================================
# Circuitos Qiskit equivalentes to the circuitos PennyLane of the training
# =============================================================================

def _weight_shape_from_params(params: Mapping) -> tuple[int, ...]:
    if not params:
        raise ValueError("params empty.")
    first = np.asarray(next(iter(params.values())))
    return tuple(first.shape)


def build_parameterized_qiskit_template(
    ansatz: str,
    weight_shape: tuple[int, ...],
    n_qubits: int,
    reupload: bool,
    measure_all_qubits: bool,
):
    """
    Builds a parameterized Qiskit template equivalent to the trained circuit.

    returns:
      circuit
      x_params
      w_params
      used_w_indices

    the embedding suportado nesta validation in hardware is AngleEmbedding/RY,
    which is the embedding used in the paper.
    """
    from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
    from qiskit.circuit import ParameterVector

    q = QuantumRegister(n_qubits, "q")
    n_clbits = n_qubits if measure_all_qubits else 1
    c = ClassicalRegister(n_clbits, "meas")
    qc = QuantumCircuit(q, c)

    x_params = ParameterVector("x", n_qubits)
    total_weights = int(np.prod(weight_shape))
    w_params = ParameterVector("w", total_weights)

    def wp(index_tuple):
        flat_idx = np.ravel_multi_index(index_tuple, weight_shape)
        return w_params[int(flat_idx)]

    n_layers = int(weight_shape[0])
    active = list(range(n_qubits))
    used_weight_indices = set()

    for layer in range(n_layers):
        if reupload or layer == 0:
            for qb in range(n_qubits):
                qc.ry(x_params[qb], q[qb])

        if ansatz == "hardware_efficient":
            for qb in range(n_qubits):
                a = np.ravel_multi_index((layer, qb, 0), weight_shape)
                b = np.ravel_multi_index((layer, qb, 1), weight_shape)
                used_weight_indices.update([int(a), int(b)])
                qc.ry(w_params[int(a)], q[qb])
                qc.rz(w_params[int(b)], q[qb])

            for qb in range(n_qubits - 1):
                qc.cx(q[qb], q[qb + 1])

        elif ansatz == "strongly_entangling":
            # in the code original, StronglyEntanglingLayers is called with the
            # layer the each iteration. immediately the range default reinicia in 1 in
            # all layer. qml.Rot(phi, theta, omega) equivale the RZ(phi),
            # RY(theta), RZ(omega).
            for qb in range(n_qubits):
                idx_phi = int(np.ravel_multi_index((layer, qb, 0), weight_shape))
                idx_theta = int(np.ravel_multi_index((layer, qb, 1), weight_shape))
                idx_omega = int(np.ravel_multi_index((layer, qb, 2), weight_shape))
                used_weight_indices.update([idx_phi, idx_theta, idx_omega])
                qc.rz(w_params[idx_phi], q[qb])
                qc.ry(w_params[idx_theta], q[qb])
                qc.rz(w_params[idx_omega], q[qb])

            if n_qubits > 1:
                for qb in range(n_qubits):
                    target = (qb + 1) % n_qubits
                    qc.cx(q[qb], q[target])

        elif ansatz == "basic_entangler":
            for qb in range(n_qubits):
                idx = int(np.ravel_multi_index((layer, qb), weight_shape))
                used_weight_indices.add(idx)
                qc.rx(w_params[idx], q[qb])

            if n_qubits == 2:
                qc.cx(q[0], q[1])
            elif n_qubits > 2:
                for qb in range(n_qubits):
                    qc.cx(q[qb], q[(qb + 1) % n_qubits])

        elif ansatz == "qaoa_inspired":
            gamma_idx = int(np.ravel_multi_index((layer, 0), weight_shape))
            beta_idx = int(np.ravel_multi_index((layer, 1), weight_shape))
            used_weight_indices.update([gamma_idx, beta_idx])

            gamma = w_params[gamma_idx]
            beta = w_params[beta_idx]

            for qb in range(n_qubits - 1):
                qc.cx(q[qb], q[qb + 1])
                qc.rz(gamma, q[qb + 1])
                qc.cx(q[qb], q[qb + 1])

            for qb in range(n_qubits):
                qc.rx(beta, q[qb])

        elif ansatz == "tree_tensor":
            new_active = []
            pair_idx = 0
            i = 0

            while i < len(active):
                if i + 1 < len(active):
                    q0, q1 = active[i], active[i + 1]

                    idx0 = int(np.ravel_multi_index((layer, pair_idx, 0), weight_shape))
                    idx1 = int(np.ravel_multi_index((layer, pair_idx, 1), weight_shape))
                    used_weight_indices.update([idx0, idx1])

                    qc.ry(w_params[idx0], q[q0])
                    qc.ry(w_params[idx1], q[q1])
                    qc.cx(q[q0], q[q1])

                    new_active.append(q0)
                    pair_idx += 1
                else:
                    new_active.append(active[i])
                i += 2

            active = new_active

        else:
            raise ValueError(f"Ansatz desconhecido: {ansatz}")

    if measure_all_qubits:
        for qb in range(n_qubits):
            qc.measure(q[qb], c[qb])
    else:
        qc.measure(q[0], c[0])

    return qc, list(x_params), list(w_params), sorted(used_weight_indices)


def bind_isa_circuit(
    isa_template,
    x_params,
    w_params,
    x_values,
    weight_array,
):
    """
    Liga values to the circuit already transpilado.

    Qiskit preserves the same Parameter objects through transpilation; the
    filtro by isa_template.parameters ensures robustez if any weight not
    seja used (by example, weights excedentes of the tree_tensor in levels final).
    """
    available = set(isa_template.parameters)
    mapping = {}

    for p, value in zip(x_params, np.asarray(x_values).ravel()):
        if p in available:
            mapping[p] = float(value)

    flat_w = np.asarray(weight_array, dtype=float).ravel()

    for p, value in zip(w_params, flat_w):
        if p in available:
            mapping[p] = float(value)

    return isa_template.assign_parameters(mapping, inplace=False)


# =============================================================================
# results of hardware
# =============================================================================

def expvals_from_counts(counts: Mapping[str, int], n_measured: int):
    """
    computes <Z> from counts of the Sampler.

    For all-q, it also returns <Z_i> for each bit and the mean magnetization, which is
    exactly the readout used in the training.
    """
    total = int(sum(int(v) for v in counts.values()))
    if total <= 0:
        raise ValueError("Counts without shots.")

    z_each = np.zeros(n_measured, dtype=float)

    for bitstring, count in counts.items():
        bits = str(bitstring).replace(" ", "")
        bits = bits.zfill(n_measured)

        # Qiskit exibe c[n-1] ... c[0]. Invertemos for obter c0, c1, ...
        ordered = list(reversed(bits[-n_measured:]))

        for i, bit in enumerate(ordered):
            z_each[i] += (1.0 if bit == "0" else -1.0) * int(count)

    z_each /= total
    z_mean = float(np.mean(z_each))

    return z_mean, z_each.tolist(), total


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(
            precision_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "precision_weighted": float(
            precision_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "recall_weighted": float(
            recall_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "f1_weighted": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "classification_report": classification_report(
            y_true,
            y_pred,
            output_dict=True,
            zero_division=0,
        ),
    }


def predict_ovr_from_scores(scores: np.ndarray, class_names: list[str]):
    """
    scolors: shape (n_samples, n_binary_estimators), values p=(<Z>+1)/2.

    Important note for the binary case:
    the OneVsRestClassifier of the sklearn trains the single estimator for two
    class_idxs. in the code original, the zip(class_idxs_, estimators_) saved this
    single weight under the first key of the dictionary, embora the estimator
    corresponds to the positive class class_idxs_[1]. Therefore, this case is handled
    explicitly this if and not confiamos in the key of the params.
    """
    n_classes = len(class_names)

    if n_classes == 2 and scores.shape[1] == 1:
        return (scores[:, 0] >= 0.5).astype(int)

    if scores.shape[1] != n_classes:
        raise RuntimeError(
            f"Number of OvR estimators ({scores.shape[1]}) differs "
            f"from the number of classes ({n_classes})."
        )

    return np.argmax(scores, axis=1).astype(int)


# =============================================================================
# IBM Quantum
# =============================================================================

def connect_ibm(token: str | None, instance: str | None):
    from qiskit_ibm_runtime import QiskitRuntimeService

    token = token or os.getenv("IBM_QUANTUM_TOKEN")
    instance = instance or os.getenv("IBM_QUANTUM_INSTANCE")

    if not token:
        raise RuntimeError(
            "Token not informado. Use --token or defina IBM_QUANTUM_TOKEN."
        )

    kwargs = {
        "channel": "ibm_quantum_platform",
        "token": token,
    }

    if instance:
        kwargs["instance"] = instance

    return QiskitRuntimeService(**kwargs)


def backend_snapshot(backend):
    """
    Captures as much information as the API exposes publicly.
    """
    out = {
        "captured_at_utc": utc_now(),
        "name": getattr(backend, "name", None),
        "backend_version": getattr(backend, "backend_version", None),
        "num_qubits": getattr(backend, "num_qubits", None),
        "online_date": getattr(backend, "online_date", None),
        "dt": getattr(backend, "dt", None),
        "dtm": getattr(backend, "dtm", None),
        "operation_names": list(getattr(backend, "operation_names", []) or []),
    }

    for method_name in ("status", "configuration", "properties", "defaults"):
        try:
            obj = getattr(backend, method_name)()
            if obj is None:
                out[method_name] = None
            elif hasattr(obj, "to_dict"):
                out[method_name] = obj.to_dict()
            else:
                out[method_name] = jsonable(obj)
        except Exception as exc:
            out[method_name] = {"_error": repr(exc)}

    try:
        coupling_map = backend.coupling_map
        if coupling_map is not None:
            out["coupling_map_edges"] = [list(edge) for edge in coupling_map.get_edges()]
        else:
            out["coupling_map_edges"] = None
    except Exception as exc:
        out["coupling_map_edges"] = {"_error": repr(exc)}

    try:
        out["target_repr"] = repr(backend.target)
    except Exception as exc:
        out["target_repr"] = {"_error": repr(exc)}

    # properties by qubit in format compacto, besides of the properties().to_dict()
    qubit_summary = []
    props = None
    try:
        props = backend.properties()
    except Exception:
        pass

    if props is not None:
        for q in range(getattr(backend, "num_qubits", 0) or 0):
            row = {"qubit": q}
            for name in ("t1", "t2", "frequency", "readout_error", "readout_length"):
                try:
                    row[name] = float(getattr(props, name)(q))
                except Exception:
                    row[name] = None
            qubit_summary.append(row)

    out["qubit_summary"] = qubit_summary
    return out


def safe_job_metadata(job):
    out = {
        "job_id": None,
        "creation_date": None,
        "primitive_id": getattr(job, "primitive_id", None),
        "session_id": getattr(job, "session_id", None),
        "instance": getattr(job, "instance", None),
        "image": getattr(job, "image", None),
        "tags": getattr(job, "tags", None),
    }

    try:
        out["job_id"] = job.job_id()
    except Exception:
        pass

    try:
        out["creation_date"] = job.creation_date
    except Exception:
        pass

    for attr in ("usage_estimation", "inputs"):
        try:
            out[attr] = getattr(job, attr)
        except Exception as exc:
            out[attr] = {"_error": repr(exc)}

    for method in ("metrics", "status"):
        try:
            out[method] = getattr(job, method)()
        except Exception as exc:
            out[method] = {"_error": repr(exc)}

    try:
        props = job.properties()
        out["job_backend_properties"] = (
            props.to_dict() if props is not None and hasattr(props, "to_dict")
            else jsonable(props)
        )
    except Exception as exc:
        out["job_backend_properties"] = {"_error": repr(exc)}

    try:
        out["logs"] = job.logs()
    except Exception as exc:
        out["logs"] = {"_error": repr(exc)}

    return out


def qasm3_dump(circuit, path: Path):
    try:
        from qiskit import qasm3
        text = qasm3.dumps(circuit)
        path.write_text(text, encoding="utf-8")
        return str(path)
    except Exception as exc:
        return {"_error": repr(exc)}


def transpilation_summary(original, isa):
    return {
        "original": {
            "num_qubits": original.num_qubits,
            "num_clbits": original.num_clbits,
            "depth": original.depth(),
            "size": original.size(),
            "count_ops": dict(original.count_ops()),
            "num_parameters": original.num_parameters,
        },
        "isa": {
            "num_qubits": isa.num_qubits,
            "num_clbits": isa.num_clbits,
            "depth": isa.depth(),
            "size": isa.size(),
            "count_ops": dict(isa.count_ops()),
            "num_parameters": isa.num_parameters,
            "layout": repr(getattr(isa, "layout", None)),
        },
    }


def run_circuits_chunked(
    sampler,
    circuits,
    shots: int,
    max_per_job: int,
    model_output_dir: Path,
    class_label: str,
):
    """
    runs circuitos already ISA in chunks and returns counts in the same order.
    """
    all_counts = []
    jobs_meta = []

    for start in range(0, len(circuits), max_per_job):
        chunk = circuits[start : start + max_per_job]
        t_submit = utc_now()

        job = sampler.run(chunk, shots=shots)

        job_id = None
        try:
            job_id = job.job_id()
        except Exception:
            pass

        print(
            f"    IBM job {job_id} | classe={class_label} | "
            f"circuitos={start}:{start + len(chunk)}"
        )

        result = job.result()
        t_finished = utc_now()

        chunk_counts = []
        for pub_result in result:
            counts = pub_result.data.meas.get_counts()
            chunk_counts.append(dict(counts))

        all_counts.extend(chunk_counts)

        meta = safe_job_metadata(job)
        meta.update(
            {
                "submitted_at_utc_client": t_submit,
                "finished_at_utc_client": t_finished,
                "class_label": class_label,
                "circuit_start_index": start,
                "n_circuits": len(chunk),
                "shots_requested": shots,
            }
        )
        jobs_meta.append(meta)

        if job_id:
            save_json(
                model_output_dir / f"job_{safe_name(job_id)}.json",
                meta,
            )

    return all_counts, jobs_meta


# =============================================================================
# Execution of the selected VQC
# =============================================================================

def validate_one_quantum_record(
    record,
    classical_record,
    dataset_bundle,
    backend,
    shots: int,
    optimization_level: int,
    seed_transpiler: int,
    max_circuits_per_job: int,
    output_root: Path,
    enable_dd: bool,
    gate_twirling: bool,
    measurement_twirling: bool,
):
    from qiskit.transpiler import generate_preset_pass_manager
    from qiskit_ibm_runtime import SamplerOptions, SamplerV2 as Sampler

    cfg = record["merged_config"]
    metrics_saved = record["metrics"]

    embedding_type = cfg.get("embedding_type", "angle")
    if embedding_type != "angle":
        raise NotImplementedError(
            "This faithful hardware-validation version supports models "
            "AngleEmbedding/RY used in the paper. the model saved with "
            f"embedding_type={embedding_type!r} foi encontrado."
        )

    params = metrics_saved.get("params")
    if not isinstance(params, Mapping) or not params:
        raise RuntimeError("The selected record does not contain params/weights.")

    n_qubits = int(cfg.get("n_qubits", dataset_bundle["X_test_q"].shape[1]))
    n_layers = int(cfg.get("n_layers"))
    ansatz = record["model_name"]
    reupload = bool(cfg.get("reupload", True))
    measure_all = bool(cfg.get("measure_all_qubits", False))

    if dataset_bundle["X_test_q"].shape[1] != n_qubits:
        raise RuntimeError(
            f"X_test tem {dataset_bundle['X_test_q'].shape[1]} features, "
            f"mas o pickle informa {n_qubits} qubits."
        )

    if backend.num_qubits < n_qubits:
        raise RuntimeError(
            f"Backend {backend.name} tem {backend.num_qubits} qubits, "
            f"but this model requires {n_qubits}."
        )

    model_id = safe_name(
        f"{record['disorder']}__{ansatz}__seed{record['random_state']}"
    )
    model_dir = output_root / model_id
    model_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"\n[{record['disorder']}] {ansatz} | "
        f"Saved F1={record['f1_weighted']:.4f} | seed={record['random_state']}"
    )

    # Ordering of estimators/weights.
    # in multiclass, keys 0..C-1 correspondem to the class_idxs.
    # In the binary case, there is only one OvR estimator.
    sorted_param_items = sorted(
        params.items(),
        key=lambda kv: int(kv[0]),
    )

    n_binary_estimators = len(sorted_param_items)
    n_classes = len(dataset_bundle["class_names"])

    if n_classes > 2 and n_binary_estimators != n_classes:
        raise RuntimeError(
            f"Expected {n_classes} OvR estimators, but params contains "
            f"{n_binary_estimators}."
        )

    if n_classes == 2 and n_binary_estimators != 1:
        raise RuntimeError(
            f"A binary problem should have 1 OvR estimator, but params contains "
            f"{n_binary_estimators}."
        )

    # Sampler options: raw hardware by default, with suppression/twirling
    # explicitly enabled through the CLI.
    options = SamplerOptions()
    try:
        options.dynamical_decoupling.enable = bool(enable_dd)
    except Exception:
        pass
    try:
        options.twirling.enable_gates = bool(gate_twirling)
        options.twirling.enable_measure = bool(measurement_twirling)
    except Exception:
        pass

    sampler = Sampler(mode=backend, options=options)

    X_test_q = dataset_bundle["X_test_q"]
    scores = np.zeros((len(X_test_q), n_binary_estimators), dtype=float)

    estimator_details = []
    all_jobs = []

    for est_idx, (saved_key, weights) in enumerate(sorted_param_items):
        weights = np.asarray(weights, dtype=float)
        weight_shape = tuple(weights.shape)

        if weight_shape[0] != n_layers:
            raise RuntimeError(
                f"Weight shape {weight_shape} is incompatible with "
                f"n_layers={n_layers}."
            )

        template, x_params, w_params, used_w_indices = (
            build_parameterized_qiskit_template(
                ansatz=ansatz,
                weight_shape=weight_shape,
                n_qubits=n_qubits,
                reupload=reupload,
                measure_all_qubits=measure_all,
            )
        )

        pm = generate_preset_pass_manager(
            backend=backend,
            optimization_level=optimization_level,
            seed_transpiler=seed_transpiler,
        )
        isa_template = pm.run(template)

        # Optional: fail early if the backend marks a used qubit/edge as faulty.
        try:
            backend.check_faulty(isa_template)
        except Exception as exc:
            raise RuntimeError(
                f"Circuito transpilado usa recurso faulty do backend: {exc}"
            ) from exc

        class_dir = model_dir / f"estimator_{est_idx}"
        class_dir.mkdir(exist_ok=True)

        qasm_original = qasm3_dump(
            template,
            class_dir / "template_logical.qasm",
        )
        qasm_isa = qasm3_dump(
            isa_template,
            class_dir / "template_isa.qasm",
        )

        transpile_info = transpilation_summary(template, isa_template)
        save_json(class_dir / "transpilation.json", transpile_info)

        bound_circuits = [
            bind_isa_circuit(
                isa_template,
                x_params,
                w_params,
                x,
                weights,
            )
            for x in X_test_q
        ]

        counts_list, jobs_meta = run_circuits_chunked(
            sampler=sampler,
            circuits=bound_circuits,
            shots=shots,
            max_per_job=max_circuits_per_job,
            model_output_dir=class_dir,
            class_label=str(saved_key),
        )
        all_jobs.extend(jobs_meta)

        z_values = []
        z_per_qubit = []
        probabilities = []

        n_measured = n_qubits if measure_all else 1

        for counts in counts_list:
            z_mean, z_each, _ = expvals_from_counts(counts, n_measured)
            p = (z_mean + 1.0) / 2.0
            z_values.append(z_mean)
            z_per_qubit.append(z_each)
            probabilities.append(p)

        scores[:, est_idx] = np.asarray(probabilities, dtype=float)

        # Counts may be grandes; remain in file own.
        save_json(class_dir / "counts.json", counts_list)

        estimator_details.append(
            {
                "estimator_index": est_idx,
                "saved_params_key": saved_key,
                "weight_shape": weight_shape,
                "n_total_saved_weights": int(weights.size),
                "used_weight_flat_indices": used_w_indices,
                "n_used_weights_in_circuit": len(used_w_indices),
                "z_expectation": z_values,
                "z_per_qubit": z_per_qubit,
                "probability_scolors": probabilities,
                "transpilation": transpile_info,
                "qasm_original": qasm_original,
                "qasm_isa": qasm_isa,
                "jobs": jobs_meta,
            }
        )

    y_pred_ibm = predict_ovr_from_scores(
        scores,
        dataset_bundle["class_names"],
    )
    y_true = dataset_bundle["y_test"]

    ibm_metrics = compute_metrics(y_true, y_pred_ibm)

    # metrics of the VQC in the simulator conforme saved in the own pickle.
    simulator_saved = {
        key: jsonable(metrics_saved.get(key))
        for key in (
            "accuracy",
            "precision_macro",
            "precision_weighted",
            "recall_macro",
            "recall_weighted",
            "f1_macro",
            "f1_weighted",
            "confusion_matrix",
            "classification_report",
            "y_true",
            "y_pred",
        )
    }

    classical_saved = None
    if classical_record is not None:
        cm = classical_record["metrics"]
        classical_saved = {
            "model_name": classical_record["model_name"],
            "config_key": classical_record["config_key"],
            "pickle_path": classical_record["pickle_path"],
            "repeat_index": classical_record["repeat_index"],
            "random_state": classical_record["random_state"],
            "accuracy": cm.get("accuracy"),
            "precision_macro": cm.get("precision_macro"),
            "precision_weighted": cm.get("precision_weighted"),
            "recall_macro": cm.get("recall_macro"),
            "recall_weighted": cm.get("recall_weighted"),
            "f1_macro": cm.get("f1_macro"),
            "f1_weighted": cm.get("f1_weighted"),
            "best_params": cm.get("best_params"),
            "confusion_matrix": cm.get("confusion_matrix"),
            "classification_report": cm.get("classification_report"),
            "y_true": cm.get("y_true"),
            "y_pred": cm.get("y_pred"),
        }

    result = {
        "experiment_finished_utc": utc_now(),
        "selection": {
            "criterion": "maximum saved f1_weighted for this disorder/ansatz",
            "disorder": record["disorder"],
            "ansatz": ansatz,
            "pickle_path": record["pickle_path"],
            "pickle_sha256": record["pickle_sha256"],
            "config_key": record["config_key"],
            "repeat_index": record["repeat_index"],
            "random_state": record["random_state"],
            "saved_best_f1_weighted": record["f1_weighted"],
        },
        "model_config": {
            "ansatz": ansatz,
            "n_qubits": n_qubits,
            "n_layers": n_layers,
            "reupload": reupload,
            "embedding_type": embedding_type,
            "measure_all_qubits": measure_all,
            "n_binary_estimators": n_binary_estimators,
            "class_names": dataset_bundle["class_names"],
        },
        "ibm": {
            "backend_name": backend.name,
            "shots_per_circuit": shots,
            "optimization_level": optimization_level,
            "seed_transpiler": seed_transpiler,
            "dynamical_decoupling": enable_dd,
            "gate_twirling": gate_twirling,
            "measurement_twirling": measurement_twirling,
            "n_test_samples": len(X_test_q),
            "n_binary_estimators": n_binary_estimators,
            "n_total_circuit_executions": len(X_test_q) * n_binary_estimators,
            "scolors": scores.tolist(),
            "y_pred": y_pred_ibm.tolist(),
            "metrics": ibm_metrics,
            "estimators": estimator_details,
            "jobs": all_jobs,
        },
        "simulator_saved": simulator_saved,
        "classical_saved_same_split": classical_saved,
        "comparison": {
            "f1_ibm": ibm_metrics["f1_weighted"],
            "f1_simulator_saved": float(metrics_saved["f1_weighted"]),
            "f1_classical_saved_same_split": (
                None
                if classical_record is None
                else float(classical_record["f1_weighted"])
            ),
            "delta_ibm_minus_simulator": (
                ibm_metrics["f1_weighted"] - float(metrics_saved["f1_weighted"])
            ),
            "delta_ibm_minus_classical": (
                None
                if classical_record is None
                else ibm_metrics["f1_weighted"]
                - float(classical_record["f1_weighted"])
            ),
        },
    }

    save_json(model_dir / "result.json", result)

    # Also save the complete pickle for later analysis in Python.
    with (model_dir / "result.pkl").open("wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)

    return result


def choose_top_k_ansatz_per_disorder(records, k=2):
    """
    Seleciona the k bestes ansätze of each dataset/disorder.

    first:
        encontra the better configuration of each ansatz
        according to the mean F1 across repetitions.

    after:
        compares the cinco ansätze of that dataset.

    Finalmente:
        keeps only the k bestes.

    the weights enviados to the QPU continuam being the weights
    of the better repeat of the configuration vencedora.
    """

    winners = choose_quantum_records(
        records,
        selection="per-disorder-ansatz",
    )

    grouped = defaultdict(list)

    for r in winners:
        grouped[
            r["disorder"]
        ].append(r)

    selected = []

    for disorder, candidates in grouped.items():

        # Ranking of the ansätze by the qualidade of sua
        # better configuration
        ranked = sorted(
            candidates,
            key=lambda r: (
                -r["configuration_mean_f1"],
                r["configuration_std_f1"],
                -r["f1_weighted"],
                r["model_name"],
            ),
        )

        # only the k bestes
        for rank, r in enumerate(
            ranked[:k],
            start=1,
        ):

            rr = dict(r)

            rr[
                "ansatz_rank_within_disorder"
            ] = rank

            rr["top_k"] = k

            selected.append(rr)

    return sorted(
        selected,
        key=lambda r: (
            r["disorder"],
            r["ansatz_rank_within_disorder"],
        ),
    )


In [ ]:
# =============================================================================
# RESILIENCE TO INTERNET DISCONNECTION / NOTEBOOK RESTART
# =============================================================================
#
# Strategy:
#   1. each chunk enviado à IBM recebe the TAG deterministic and single.
#   2. before of submeter the new job, procuramos:
#        the) checkpoint local with job_id;
#        b) the own IBM Quantum Service by this TAG.
#   3. the job_id is saved imediatamente after the submission.
#   4. Counts from each completed chunk are saved immediately.
#   5. If the internet connection drops while the job is QUEUED/RUNNING, simply run
#      the cell again: the existing job is recovered, NOT resubmitted.
#   6. If the kernel terminates after submit and before saving the job_id, the TAG
#      allows localizar the job directly in the IBM.
#
# This eliminates the need to restart the experiment from scratch.

import uuid


def atomic_save_json(path: Path, obj: Any):
    """Atomic write to avoid a truncated JSON file if the kernel/process terminates."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(jsonable(obj), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def atomic_pickle_dump(path: Path, obj: Any):
    """Atomic pickle checkpoint."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


def load_json_if_exists(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {} if default is None else default


def stable_short_hash(*parts, length=16):
    raw = "||".join(str(p) for p in parts).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:length]


def get_or_create_run_id(output_root: Path, backend_name: str, selection_mode: str):
    """
    keeps the same run_id to the reabrir/reexecutar the notebook in the same OUTPUT_DIR.
    this is essencial for reencontrar jobs remotos by the tag.
    """
    state_path = Path(output_root) / "resume_state.json"
    state = load_json_if_exists(state_path, default={})

    if state.get("run_id"):
        return state["run_id"]

    run_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-"
        + uuid.uuid4().hex[:8]
    )

    state = {
        "run_id": run_id,
        "created_at_utc": utc_now(),
        "backend_name": backend_name,
        "selection_mode": selection_mode,
        "models": {},
    }
    atomic_save_json(state_path, state)
    return run_id


def update_resume_state(output_root: Path, updater):
    """
    loads resume_state.json, applies updater(state) and saved atomicamente.
    """
    path = Path(output_root) / "resume_state.json"
    state = load_json_if_exists(path, default={})
    updater(state)
    atomic_save_json(path, state)
    return state


def make_job_tag(run_id, model_id, estimator_idx, chunk_start):
    """
    Deterministic tag. If the kernel terminates before saving job_id locally,
    the tag remains existindo in the IBM and allows reencontrar the job.
    """
    short = stable_short_hash(
        run_id, model_id, estimator_idx, chunk_start, length=18
    )
    return f"vqcval-{short}"


def find_remote_job_by_tag(service, backend_name, job_tag):
    """
    Searches IBM for an already-submitted job with the exact tag.
    returns the job more recente, or None.
    """
    try:
        jobs = service.jobs(
            limit=None,
            backend_name=backend_name,
            job_tags=[job_tag],
            descending=True,
        )
        return jobs[0] if jobs else None
    except Exception as exc:
        print(f"      [warning] Could not search for remote tag {job_tag}: {exc}")
        return None


def get_job_status_text(job):
    try:
        status = job.status()
        return str(getattr(status, "name", status))
    except Exception:
        return "UNKNOWN"


def wait_for_job_with_reconnect(
    service,
    job_id,
    *,
    poll_seconds=30,
    max_local_wait_seconds=None,
):
    """
    Waits for an existing job without resubmitting it.

    - reconnects via service.job(job_id) the each iteration.
    - if there is error of network, waits and tries again.
    - max_local_wait_seconds=None => waits indefinitely.
      this not altera the job remoto; only controla quanto the notebook waits.
    """
    started = time.time()
    last_status = None

    while True:
        if (
            max_local_wait_seconds is not None
            and time.time() - started > max_local_wait_seconds
        ):
            raise TimeoutError(
                f"Maximum local wait time exceeded while waiting for job {job_id}. "
                "The job was NOT canceled. Run the cell again later; it will be recovered."
            )

        try:
            job = service.job(job_id)
            status = get_job_status_text(job)

            if status != last_status:
                print(f"      job {job_id}: {status}")
                last_status = status

            if job.in_final_state():
                if job.errored():
                    msg = None
                    try:
                        msg = job.error_message()
                    except Exception:
                        pass
                    raise RuntimeError(f"IBM job {job_id} terminou com ERROR: {msg}")

                if job.cancelled():
                    raise RuntimeError(f"IBM job {job_id} foi CANCELLED.")

                return job

            time.sleep(poll_seconds)

        except (KeyboardInterrupt, SystemExit):
            raise

        except RuntimeError:
            raise

        except Exception as exc:
            print(
                f"      [network] Failed to query {job_id}: {exc}. "
                f"Tentando novamente em {poll_seconds}s..."
            )
            time.sleep(poll_seconds)


def extract_counts_from_job_result(job):
    result = job.result()
    counts_list = []
    for pub_result in result:
        counts = pub_result.data.meas.get_counts()
        counts_list.append(dict(counts))
    return counts_list


def run_circuits_chunked_resumable(
    *,
    service,
    backend,
    circuits,
    shots: int,
    max_per_job: int,
    model_output_dir: Path,
    class_label: str,
    sampler_options,
    run_id: str,
    model_id: str,
    estimator_idx: int,
    poll_seconds: int = 30,
    max_local_wait_seconds=None,
):
    """
    Resumable version of run_circuits_chunked().

    for each chunk:
      - if counts already existem: reutiliza;
      - if job_id existe in the checkpoint: recupera job in the IBM;
      - if job_id local not existe: procura job remoto by the TAG;
      - only if not encontrar nada: submete new job;
      - saved job_id IMEDIATAMENTE;
      - aguarda/reconnects;
      - saved counts IMEDIATAMENTE.

    returns counts in the order original and metadata of the jobs.
    """
    from qiskit_ibm_runtime import SamplerV2 as Sampler

    model_output_dir = Path(model_output_dir)
    model_output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = model_output_dir / "chunks_checkpoint.json"

    checkpoint = load_json_if_exists(
        checkpoint_path,
        default={"version": 2, "chunks": {}},
    )
    checkpoint.setdefault("version", 2)
    checkpoint.setdefault("chunks", {})

    all_counts = []
    jobs_meta = []

    for start in range(0, len(circuits), max_per_job):
        chunk = circuits[start : start + max_per_job]
        chunk_key = f"{start}:{start + len(chunk)}"

        tag = make_job_tag(
            run_id=run_id,
            model_id=model_id,
            estimator_idx=estimator_idx,
            chunk_start=start,
        )

        chunk_state = checkpoint["chunks"].setdefault(
            chunk_key,
            {
                "start": start,
                "n_circuits": len(chunk),
                "shots": shots,
                "job_tag": tag,
                "job_id": None,
                "status": "NOT_SUBMITTED",
                "counts_file": None,
            },
        )

        # --------------------------------------------------------------
        # 1) Chunk already finalizado localmente => not consulta/submete nada.
        # --------------------------------------------------------------
        counts_file = chunk_state.get("counts_file")
        if (
            chunk_state.get("status") == "DONE"
            and counts_file
            and Path(counts_file).exists()
        ):
            print(
                f"    [resume] classe={class_label} chunk={chunk_key}: "
                "counts already saved; skipping."
            )
            counts_list = json.loads(Path(counts_file).read_text(encoding="utf-8"))
            all_counts.extend(counts_list)

            meta_file = chunk_state.get("job_meta_file")
            if meta_file and Path(meta_file).exists():
                jobs_meta.append(
                    json.loads(Path(meta_file).read_text(encoding="utf-8"))
                )
            else:
                jobs_meta.append({
                    "job_id": chunk_state.get("job_id"),
                    "job_tag": tag,
                    "status": "DONE",
                    "recovered_from_checkpoint": True,
                })
            continue

        # --------------------------------------------------------------
        # 2) tries recuperar job_id local.
        # --------------------------------------------------------------
        job = None
        job_id = chunk_state.get("job_id")

        if job_id:
            try:
                job = service.job(job_id)
                print(
                    f"    [resume] Recovered local job {job_id} "
                    f"(classe={class_label}, chunk={chunk_key})."
                )
            except Exception as exc:
                print(
                    f"    [resume] Job ID {job_id} could not yet be "
                    f"recovered: {exc}"
                )

        # --------------------------------------------------------------
        # 3) If the kernel terminated between submit and saving the job_id,
        #    procura in the plataforma IBM by the tag deterministic.
        # --------------------------------------------------------------
        if job is None:
            remote_job = find_remote_job_by_tag(
                service,
                backend.name,
                tag,
            )
            if remote_job is not None:
                job = remote_job
                job_id = job.job_id()

                print(
                    f"    [recovery IBM] Job encontrado por TAG: {job_id} "
                    f"(classe={class_label}, chunk={chunk_key})."
                )

                chunk_state["job_id"] = job_id
                chunk_state["status"] = get_job_status_text(job)
                chunk_state["recovered_by_tag"] = True
                chunk_state["recovered_at_utc"] = utc_now()
                atomic_save_json(checkpoint_path, checkpoint)

        # --------------------------------------------------------------
        # 4) only here is permitido submeter the job new.
        # --------------------------------------------------------------
        if job is None:
            options = sampler_options

            # Each chunk receives its own tags, enabling remote recovery.
            try:
                options.environment.job_tags = [
                    "vqc-hardware-validation",
                    f"run-{run_id}",
                    tag,
                ]
            except Exception as exc:
                raise RuntimeError(
                    "Your version of qiskit-ibm-runtime did not accept "
                    "environment.job_tags. Atualize the pacote before of the execution "
                    "to ensure safe job recovery."
                ) from exc

            sampler = Sampler(mode=backend, options=options)

            print(
                f"    [submit] NOVO job | classe={class_label} "
                f"| chunk={chunk_key} | tag={tag}"
            )

            t_submit = utc_now()
            job = sampler.run(chunk, shots=shots)
            job_id = job.job_id()

            # CRITICAL: save immediately after obtaining job_id.
            chunk_state.update({
                "job_id": job_id,
                "job_tag": tag,
                "status": "SUBMITTED",
                "submitted_at_utc_client": t_submit,
                "submitted_new": True,
            })
            atomic_save_json(checkpoint_path, checkpoint)

            # also atualiza manifest geral.
            def _upd(state):
                state.setdefault("jobs", {})
                state["jobs"][tag] = {
                    "job_id": job_id,
                    "backend": backend.name,
                    "model_id": model_id,
                    "estimator_idx": estimator_idx,
                    "class_label": class_label,
                    "chunk": chunk_key,
                    "submitted_at_utc": t_submit,
                }

            update_resume_state(model_output_dir.parent.parent, _upd)

            print(f"      job_id saved: {job_id}")

        # --------------------------------------------------------------
        # 5) Wait. A network interruption does not trigger a new submission.
        # --------------------------------------------------------------
        chunk_state["status"] = get_job_status_text(job)
        atomic_save_json(checkpoint_path, checkpoint)

        job = wait_for_job_with_reconnect(
            service,
            job_id,
            poll_seconds=poll_seconds,
            max_local_wait_seconds=max_local_wait_seconds,
        )

        # --------------------------------------------------------------
        # 6) Job terminou: captura result and saved before of continuar.
        # --------------------------------------------------------------
        counts_list = extract_counts_from_job_result(job)

        counts_path = model_output_dir / f"counts_chunk_{start:05d}.json"
        atomic_save_json(counts_path, counts_list)

        meta = safe_job_metadata(job)
        meta.update({
            "job_tag": tag,
            "class_label": class_label,
            "circuit_start_index": start,
            "n_circuits": len(chunk),
            "shots_requested": shots,
            "recovered_or_resumed": bool(
                chunk_state.get("recovered_by_tag")
                or not chunk_state.get("submitted_new", False)
            ),
        })

        meta_path = model_output_dir / f"job_{safe_name(job_id)}.json"
        atomic_save_json(meta_path, meta)

        chunk_state.update({
            "status": "DONE",
            "finished_at_utc_client": utc_now(),
            "counts_file": str(counts_path.resolve()),
            "job_meta_file": str(meta_path.resolve()),
        })
        atomic_save_json(checkpoint_path, checkpoint)

        all_counts.extend(counts_list)
        jobs_meta.append(meta)

    # file consolidado by estimator.
    atomic_save_json(model_output_dir / "counts.json", all_counts)
    return all_counts, jobs_meta


def validate_one_quantum_record_resumable(
    *,
    service,
    run_id: str,
    record,
    classical_record,
    dataset_bundle,
    backend,
    shots: int,
    optimization_level: int,
    seed_transpiler: int,
    max_circuits_per_job: int,
    output_root: Path,
    enable_dd: bool,
    gate_twirling: bool,
    measurement_twirling: bool,
    poll_seconds: int = 30,
    max_local_wait_seconds=None,
):
    """
    Equivalent to the previous validation, but with a checkpoint for each estimator/chunk.

    if result.json already existe for the model, ele is loaded directly.
    """
    from qiskit.transpiler import generate_preset_pass_manager
    from qiskit_ibm_runtime import SamplerOptions

    cfg = record["merged_config"]
    metrics_saved = record["metrics"]

    embedding_type = cfg.get("embedding_type", "angle")
    if embedding_type != "angle":
        raise NotImplementedError(
            "the validation atual in hardware suporta AngleEmbedding/RY."
        )

    params = metrics_saved.get("params")
    if not isinstance(params, Mapping) or not params:
        raise RuntimeError("The selected record does not contain params/weights.")

    n_qubits = int(cfg.get("n_qubits", dataset_bundle["X_test_q"].shape[1]))
    n_layers = int(cfg.get("n_layers"))
    ansatz = record["model_name"]
    reupload = bool(cfg.get("reupload", True))
    measure_all = bool(cfg.get("measure_all_qubits", False))

    model_id = safe_name(
        f"{record['disorder']}__{ansatz}__seed{record['random_state']}"
    )
    model_dir = Path(output_root) / model_id
    model_dir.mkdir(parents=True, exist_ok=True)

    final_result_json = model_dir / "result.json"
    final_result_pkl = model_dir / "result.pkl"

    # Entire model already completed.
    if final_result_json.exists():
        print(
            f"\n[resume] {record['disorder']} | {ansatz}: "
            "result final already existe; model skipped."
        )
        return json.loads(final_result_json.read_text(encoding="utf-8"))

    if dataset_bundle["X_test_q"].shape[1] != n_qubits:
        raise RuntimeError(
            f"X_test tem {dataset_bundle['X_test_q'].shape[1]} features, "
            f"mas o pickle informa {n_qubits} qubits."
        )

    if backend.num_qubits < n_qubits:
        raise RuntimeError(
            f"Backend {backend.name} tem {backend.num_qubits} qubits, "
            f"but the model requires {n_qubits}."
        )

    print(
        f"\n[{record['disorder']}] {ansatz} | "
        f"Saved F1={record['f1_weighted']:.4f} | "
        f"seed={record['random_state']}"
    )

    sorted_param_items = sorted(
        params.items(),
        key=lambda kv: int(kv[0]),
    )

    n_binary_estimators = len(sorted_param_items)
    n_classes = len(dataset_bundle["class_names"])

    if n_classes > 2 and n_binary_estimators != n_classes:
        raise RuntimeError(
            f"Expected {n_classes} OvR estimators, but there are "
            f"{n_binary_estimators}."
        )
    if n_classes == 2 and n_binary_estimators != 1:
        raise RuntimeError(
            f"A binary problem should have 1 OvR estimator, but there are "
            f"{n_binary_estimators}."
        )

    options = SamplerOptions()
    try:
        options.dynamical_decoupling.enable = bool(enable_dd)
    except Exception:
        pass
    try:
        options.twirling.enable_gates = bool(gate_twirling)
        options.twirling.enable_measure = bool(measurement_twirling)
    except Exception:
        pass

    X_test_q = dataset_bundle["X_test_q"]
    scores = np.zeros((len(X_test_q), n_binary_estimators), dtype=float)

    estimator_details = []
    all_jobs = []

    for est_idx, (saved_key, weights) in enumerate(sorted_param_items):
        weights = np.asarray(weights, dtype=float)
        weight_shape = tuple(weights.shape)

        if weight_shape[0] != n_layers:
            raise RuntimeError(
                f"Shape {weight_shape} is incompatible with n_layers={n_layers}."
            )

        template, x_params, w_params, used_w_indices = (
            build_parameterized_qiskit_template(
                ansatz=ansatz,
                weight_shape=weight_shape,
                n_qubits=n_qubits,
                reupload=reupload,
                measure_all_qubits=measure_all,
            )
        )

        pm = generate_preset_pass_manager(
            backend=backend,
            optimization_level=optimization_level,
            seed_transpiler=seed_transpiler,
        )
        isa_template = pm.run(template)

        try:
            backend.check_faulty(isa_template)
        except Exception as exc:
            raise RuntimeError(
                f"Circuito transpilado usa recurso faulty: {exc}"
            ) from exc

        class_dir = model_dir / f"estimator_{est_idx}"
        class_dir.mkdir(exist_ok=True)

        qasm_original = qasm3_dump(
            template,
            class_dir / "template_logical.qasm",
        )
        qasm_isa = qasm3_dump(
            isa_template,
            class_dir / "template_isa.qasm",
        )

        transpile_info = transpilation_summary(template, isa_template)
        atomic_save_json(class_dir / "transpilation.json", transpile_info)

        bound_circuits = [
            bind_isa_circuit(
                isa_template,
                x_params,
                w_params,
                x,
                weights,
            )
            for x in X_test_q
        ]

        counts_list, jobs_meta = run_circuits_chunked_resumable(
            service=service,
            backend=backend,
            circuits=bound_circuits,
            shots=shots,
            max_per_job=max_circuits_per_job,
            model_output_dir=class_dir,
            class_label=str(saved_key),
            sampler_options=options,
            run_id=run_id,
            model_id=model_id,
            estimator_idx=est_idx,
            poll_seconds=poll_seconds,
            max_local_wait_seconds=max_local_wait_seconds,
        )
        all_jobs.extend(jobs_meta)

        z_values = []
        z_per_qubit = []
        probabilities = []
        n_measured = n_qubits if measure_all else 1

        for counts in counts_list:
            z_mean, z_each, _ = expvals_from_counts(counts, n_measured)
            p = (z_mean + 1.0) / 2.0
            z_values.append(z_mean)
            z_per_qubit.append(z_each)
            probabilities.append(p)

        scores[:, est_idx] = np.asarray(probabilities, dtype=float)

        detail = {
            "estimator_index": est_idx,
            "saved_params_key": saved_key,
            "weight_shape": weight_shape,
            "n_total_saved_weights": int(weights.size),
            "used_weight_flat_indices": used_w_indices,
            "n_used_weights_in_circuit": len(used_w_indices),
            "z_expectation": z_values,
            "z_per_qubit": z_per_qubit,
            "probability_scolors": probabilities,
            "transpilation": transpile_info,
            "qasm_original": qasm_original,
            "qasm_isa": qasm_isa,
            "jobs": jobs_meta,
        }
        estimator_details.append(detail)

        # Model checkpoint after each completed OvR estimator.
        atomic_save_json(
            model_dir / "partial_model_state.json",
            {
                "run_id": run_id,
                "model_id": model_id,
                "disorder": record["disorder"],
                "ansatz": ansatz,
                "completed_estimators": est_idx + 1,
                "n_estimators": n_binary_estimators,
                "updated_at_utc": utc_now(),
                "jobs": all_jobs,
            },
        )

    y_pred_ibm = predict_ovr_from_scores(
        scores,
        dataset_bundle["class_names"],
    )
    y_true = dataset_bundle["y_test"]

    ibm_metrics = compute_metrics(y_true, y_pred_ibm)

    simulator_saved = {
        key: jsonable(metrics_saved.get(key))
        for key in (
            "accuracy",
            "precision_macro",
            "precision_weighted",
            "recall_macro",
            "recall_weighted",
            "f1_macro",
            "f1_weighted",
            "confusion_matrix",
            "classification_report",
            "y_true",
            "y_pred",
        )
    }

    classical_saved = None
    if classical_record is not None:
        cm = classical_record["metrics"]
        classical_saved = {
            "model_name": classical_record["model_name"],
            "config_key": classical_record["config_key"],
            "pickle_path": classical_record["pickle_path"],
            "repeat_index": classical_record["repeat_index"],
            "random_state": classical_record["random_state"],
            "accuracy": cm.get("accuracy"),
            "precision_macro": cm.get("precision_macro"),
            "precision_weighted": cm.get("precision_weighted"),
            "recall_macro": cm.get("recall_macro"),
            "recall_weighted": cm.get("recall_weighted"),
            "f1_macro": cm.get("f1_macro"),
            "f1_weighted": cm.get("f1_weighted"),
            "best_params": cm.get("best_params"),
            "confusion_matrix": cm.get("confusion_matrix"),
            "classification_report": cm.get("classification_report"),
            "y_true": cm.get("y_true"),
            "y_pred": cm.get("y_pred"),
        }

    result = {
        "experiment_finished_utc": utc_now(),
        "run_id": run_id,
        "selection": {
            "criterion": "selected quantum configuration/repeat from saved results",
            "disorder": record["disorder"],
            "ansatz": ansatz,
            "pickle_path": record["pickle_path"],
            "pickle_sha256": record["pickle_sha256"],
            "config_key": record["config_key"],
            "repeat_index": record["repeat_index"],
            "random_state": record["random_state"],
            "saved_best_f1_weighted": record["f1_weighted"],
        },
        "model_config": {
            "ansatz": ansatz,
            "n_qubits": n_qubits,
            "n_layers": n_layers,
            "reupload": reupload,
            "embedding_type": embedding_type,
            "measure_all_qubits": measure_all,
            "n_binary_estimators": n_binary_estimators,
            "class_names": dataset_bundle["class_names"],
        },
        "ibm": {
            "backend_name": backend.name,
            "shots_per_circuit": shots,
            "optimization_level": optimization_level,
            "seed_transpiler": seed_transpiler,
            "dynamical_decoupling": enable_dd,
            "gate_twirling": gate_twirling,
            "measurement_twirling": measurement_twirling,
            "n_test_samples": len(X_test_q),
            "n_binary_estimators": n_binary_estimators,
            "n_total_circuit_executions": len(X_test_q) * n_binary_estimators,
            "scolors": scores.tolist(),
            "y_pred": y_pred_ibm.tolist(),
            "metrics": ibm_metrics,
            "estimators": estimator_details,
            "jobs": all_jobs,
        },
        "simulator_saved": simulator_saved,
        "classical_saved_same_split": classical_saved,
        "comparison": {
            "f1_ibm": ibm_metrics["f1_weighted"],
            "f1_simulator_saved": float(metrics_saved["f1_weighted"]),
            "f1_classical_saved_same_split": (
                None if classical_record is None
                else float(classical_record["f1_weighted"])
            ),
            "delta_ibm_minus_simulator": (
                ibm_metrics["f1_weighted"]
                - float(metrics_saved["f1_weighted"])
            ),
            "delta_ibm_minus_classical": (
                None if classical_record is None
                else ibm_metrics["f1_weighted"]
                - float(classical_record["f1_weighted"])
            ),
        },
    }

    atomic_save_json(final_result_json, result)
    atomic_pickle_dump(final_result_pkl, result)

    def _mark_done(state):
        state.setdefault("models", {})
        state["models"][model_id] = {
            "status": "DONE",
            "result_json": str(final_result_json.resolve()),
            "updated_at_utc": utc_now(),
        }

    update_resume_state(output_root, _mark_done)

    return result


def recover_ibm_jobs_for_this_run(
    service,
    output_root: Path,
    backend_name: str,
):
    """
    Shows the jobs that have already been launched by this experiment.

    first uses the IDs saved localmente; after consulta the IBM by the tag of the run.
    """
    output_root = Path(output_root)
    state = load_json_if_exists(output_root / "resume_state.json", default={})
    run_id = state.get("run_id")

    if not run_id:
        return pd.DataFrame()

    rows = []
    seen = set()

    # Jobs conhecidos localmente.
    for tag, meta in state.get("jobs", {}).items():
        job_id = meta.get("job_id")
        if not job_id or job_id in seen:
            continue

        try:
            job = service.job(job_id)
            status = get_job_status_text(job)
        except Exception as exc:
            status = f"UNREACHABLE: {exc}"

        rows.append({
            "job_id": job_id,
            "job_tag": tag,
            "status": status,
            "backend": meta.get("backend"),
            "model_id": meta.get("model_id"),
            "estimator_idx": meta.get("estimator_idx"),
            "chunk": meta.get("chunk"),
            "source": "local checkpoint",
        })
        seen.add(job_id)

    # Additional recovery directly from IBM.
    try:
        remote_jobs = service.jobs(
            limit=None,
            backend_name=backend_name,
            job_tags=[f"run-{run_id}"],
            descending=True,
        )

        for job in remote_jobs:
            job_id = job.job_id()
            if job_id in seen:
                continue

            rows.append({
                "job_id": job_id,
                "job_tag": None,
                "status": get_job_status_text(job),
                "backend": backend_name,
                "model_id": None,
                "estimator_idx": None,
                "chunk": None,
                "source": "IBM tag recovery",
            })
            seen.add(job_id)

    except Exception as exc:
        print(f"[warning] Remote search for run jobs failed: {exc}")

    return pd.DataFrame(rows)


### Checkpointing and automatic recovery

From this cell onward, every submitted job receives a **deterministic tag**, and its
`job_id` is saved immediately. If the connection drops or the kernel is restarted,
the notebook first checks the local checkpoint and, if necessary, queries IBM
directly using the tag. Jobs that have already been submitted **are not resubmitted**.

Completed counts are also persisted for each chunk. Therefore, when Section 9 is
run again, the notebook resumes from the first chunk that has not yet been completed.


## 2. Experiment configuration

In most runs, you only need to edit this cell.

**Recommendation:** store the token in an environment variable instead of writing it directly in the notebook.


In [ ]:
from pathlib import Path
import os

# -------------------------------------------------------------------------
# files
# -------------------------------------------------------------------------
PICKLE_PATTERNS = [
    # can use the or more files/globs:
    # "/home/seu_usuario/results/*.pkl",
    "outputs/*.pkl",
]

CSV_PATH = "../dataset/dataset_final_mental_health.csv"

OUTPUT_DIR = Path("ibm_validation")

# -------------------------------------------------------------------------
# IBM QUANTUM
# -------------------------------------------------------------------------
# Preferred:
# the.environ["IBM_QUANTUM_TOKEN"] = "SEU_TOKEN"
# the.environ["IBM_QUANTUM_INSTANCE"] = "SEU_CRN"


LOGIN_IBM = "fernandoneto7"

if LOGIN_IBM == "fernando":
    IBM_TOKEN = "######OMITTED##########"
    IBM_INSTANCE = "open-instance"
elif LOGIN_IBM == "fernandoneto7":
    IBM_TOKEN =  "######OMITTED##########"
    IBM_INSTANCE = "open-instance"


# EXACT name of the processor that you always want to use.
IBM_BACKEND = "ibm_fez"

# -------------------------------------------------------------------------
# selection of the models
# -------------------------------------------------------------------------
# "per-disorder-ansatz":
#   seleciona 1 repeat of highest F1 for each ansatz in each disorder.
#
# "per-disorder":
#   seleciona only the better VQC geral of each disorder.
SELECTION_MODE = "top-k-per-disorder"
TOP_K_ANSATZ_PER_DISORDER = 2

# -------------------------------------------------------------------------
# execution in the QPU
# -------------------------------------------------------------------------
SHOTS = 1024
TEST_SIZE = 0.30
OPTIMIZATION_LEVEL = 3
SEED_TRANSPILER = 42

# Limita quantidade of circuitos by job IBM.
MAX_CIRCUITS_PER_JOB = 100

# Recovery/resume:
POLL_SECONDS = 30

# None = the notebook may wait indefinitely in the queue.
# Example: 3600 makes the cell stop waiting after 1 hour, WITHOUT canceling the job.
# When the cell is run again, it recovers the same job_id.
MAX_LOCAL_WAIT_SECONDS = None

# by default, deixe False for obter the execution "raw".
DYNAMICAL_DECOUPLING = False
GATE_TWIRLING = False
MEASUREMENT_TWIRLING = False

# -------------------------------------------------------------------------
# columns of the dataset
# -------------------------------------------------------------------------
COLUMN_MAIN = "main.disorder"
COLUMN_SPECIFIC = "specific.disorder"
GROUP_CONTROL = "Healthy control"
COLUMNS_EXCLUDE = ("sex_M",)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("configuration carregada.")


## 3. Connect to IBM Quantum and list available QPUs

Run this cell before deciding the value of `IBM_BACKEND`.

It only queries your account and **does not submit any job**.


In [ ]:
service = connect_ibm(
    token=IBM_TOKEN,
    instance=IBM_INSTANCE,
)

backends = service.backends(
    operational=True,
    simulator=False,
)

rows = []
for b in backends:
    try:
        status = b.status()
        pending = status.pending_jobs
        operational = status.operational
    except Exception:
        pending = None
        operational = None

    rows.append({
        "name": b.name,
        "num_qubits": b.num_qubits,
        "pending_jobs": pending,
        "operational": operational,
        "backend_version": getattr(b, "backend_version", None),
    })

df_backends = pd.DataFrame(rows)

if not df_backends.empty:
    df_backends = df_backends.sort_values(
        ["pending_jobs", "name"],
        na_position="last",
    )

display(df_backends)


## 4. Load the pickle files and select the weights with the highest F1

This step does not use IBM. It shows exactly which weights/repetitions will be sent to the hardware.

By default, selection is performed for each **disorder + ansatz**, using the highest saved `f1_weighted` across all provided pickle files.


In [ ]:
pickle_paths = resolve_pickle_paths(PICKLE_PATTERNS)
records = list(iter_records(pickle_paths))

print(f"{len(pickle_paths)} pickle(s) encontrado(s).")
print(f"{len(records)} model/repetition records found.")

selected = choose_top_k_ansatz_per_disorder(
    records,
    k=TOP_K_ANSATZ_PER_DISORDER,
)

df_selected = selection_table(
    selected,
    records,
)

display(df_selected)

print(
    f"Total selected for hardware: "
    f"{len(selected)}"
)

selection_csv = OUTPUT_DIR / "selected_models.csv"
df_selected.to_csv(selection_csv, index=False)

print(f"\nSelection saved to: {selection_csv}")


## 5. Load the original dataset and verify test-set reproduction

Before spending QPU time, this step reconstructs the selected splits and requires `y_test` to be exactly equal to the `y_true` saved in the pickle file.

If there is any mismatch, validation must be stopped.


In [ ]:
df = pd.read_csv(CSV_PATH)

reconstruction_checks = []

for qrec in selected:
    try:
        bundle = reconstruct_test_set(
            df=df,
            record=qrec,
            test_size=TEST_SIZE,
            column_main=COLUMN_MAIN,
            column_specific=COLUMN_SPECIFIC,
            group_control=GROUP_CONTROL,
            columns_exclude=COLUMNS_EXCLUDE,
        )

        reconstruction_checks.append({
            "disorder": qrec["disorder"],
            "model": qrec["model_name"],
            "random_state": qrec["random_state"],
            "n_test": len(bundle["y_test"]),
            "n_features_quantum": bundle["X_test_q"].shape[1],
            "class_idxs": bundle["class_names"],
            "status": "OK",
        })

    except Exception as exc:
        reconstruction_checks.append({
            "disorder": qrec["disorder"],
            "model": qrec["model_name"],
            "random_state": qrec["random_state"],
            "status": f"ERROR: {exc}",
        })

df_reconstruction = pd.DataFrame(reconstruction_checks)
display(df_reconstruction)

if not (df_reconstruction["status"] == "OK").all():
    raise RuntimeError(
        "At least one split could not be reproduced exactly. "
        "Corrija before of enviar jobs à IBM."
    )

print("All selected test sets were reconstructed correctly.")


## 6. Fix the QPU and save an initial snapshot

The processor is selected explicitly through `IBM_BACKEND`.

**There is no automatic fallback**: if that QPU is not operational, the notebook stops instead of switching processors.


In [ ]:
backend = service.backend(IBM_BACKEND)

if getattr(backend, "simulator", False):
    raise RuntimeError(f"{IBM_BACKEND} is a simulator; select a real QPU.")

status = backend.status()

if not status.operational:
    raise RuntimeError(
        f"The fixed backend {IBM_BACKEND} is not operational. "
        "No other QPU will be selected automatically."
    )

print(f"Selected QPU: {backend.name}")
print(f"Qubits: {backend.num_qubits}")
print(f"Jobs pendentes: {getattr(status, 'pending_jobs', None)}")

backend_info_before = backend_snapshot(backend)
save_json(
    OUTPUT_DIR / "backend_snapshot_before.json",
    backend_info_before,
)

print(f"Initial snapshot saved to {OUTPUT_DIR / 'backend_snapshot_before.json'}")


# ID persistent of this campanha of validation.
RUN_ID = get_or_create_run_id(
    OUTPUT_DIR,
    backend_name=IBM_BACKEND,
    selection_mode=SELECTION_MODE,
)
print(f"RUN_ID persistente: {RUN_ID}")


## 7. Reproducibility manifest

Stores software versions, file hashes, QPU configuration, and execution parameters. The token is not stored.


In [ ]:
run_manifest = {
    "started_at_utc": utc_now(),
    "hostname": socket.gethostname(),
    "platform": platform.platform(),
    "python": sys.version,
    "package_versions": package_versions(),

    "input": {
        "csv": str(Path(CSV_PATH).resolve()),
        "csv_sha256": sha256_file(CSV_PATH),
        "pickles": [
            {
                "path": str(p),
                "sha256": sha256_file(p),
            }
            for p in pickle_paths
        ],
    },

    "ibm": {
        "channel": "ibm_quantum_platform",
        "instance_provided": bool(IBM_INSTANCE),
        "token_stored": False,
        "fixed_backend": IBM_BACKEND,
        "shots": SHOTS,
        "optimization_level": OPTIMIZATION_LEVEL,
        "seed_transpiler": SEED_TRANSPILER,
        "dynamical_decoupling": DYNAMICAL_DECOUPLING,
        "gate_twirling": GATE_TWIRLING,
        "measurement_twirling": MEASUREMENT_TWIRLING,
    },

    "selection": SELECTION_MODE,
    "backend_snapshot_before": backend_info_before,
}

save_json(
    OUTPUT_DIR / "run_manifest_start.json",
    run_manifest,
)

print("manifest initial saved.")


## 8. Optional test with a single model

It is **strongly recommended** to run only one configuration first in order to validate:
- circuit compatibility;
- transpilation;
- result format;
- QPU access;
- expected cost/time.

Change `TEST_INDEX` to select another row from the selection table.


In [ ]:
TEST_INDEX = 0

qrec = selected[TEST_INDEX]
classical = choose_best_classical_same_split(
    qrec,
    records,
)

bundle = reconstruct_test_set(
    df=df,
    record=qrec,
    test_size=TEST_SIZE,
    column_main=COLUMN_MAIN,
    column_specific=COLUMN_SPECIFIC,
    group_control=GROUP_CONTROL,
    columns_exclude=COLUMNS_EXCLUDE,
)

print("The following will be executed:")
print("  disorder:", qrec["disorder"])
print("  ansatz:", qrec["model_name"])
print("  F1 saved:", qrec["f1_weighted"])
print("  repeat:", qrec["repeat_index"])
print("  random_state:", qrec["random_state"])
print("  classical:", None if classical is None else classical["model_name"])

# DESCOMENTE for enviar the job real:
#
# test_result = validate_one_quantum_record_resumable(
#     service=service,
#     run_id=RUN_ID,
#     record=qrec,
#     classical_record=classical,
#     dataset_bundle=bundle,
#     backend=backend,
#     shots=SHOTS,
#     optimization_level=OPTIMIZATION_LEVEL,
#     seed_transpiler=SEED_TRANSPILER,
#     max_circuits_per_job=MAX_CIRCUITS_PER_JOB,
#     output_root=OUTPUT_DIR,
#     enable_dd=DYNAMICAL_DECOUPLING,
#     gate_twirling=GATE_TWIRLING,
#     measurement_twirling=MEASUREMENT_TWIRLING,
#     poll_seconds=POLL_SECONDS,
#     max_local_wait_seconds=MAX_LOCAL_WAIT_SECONDS,
# )
#
# display(pd.DataFrame([{
#     "disorder": test_result["selection"]["disorder"],
#     "model": test_result["selection"]["ansatz"],
#     "F1 simulator": test_result["comparison"]["f1_simulator_saved"],
#     "F1 IBM": test_result["comparison"]["f1_ibm"],
#     "F1 classical": test_result["comparison"]["f1_classical_saved_same_split"],
# }]))


## 9. Run/resume all selected models on the QPU

⚠️ This cell may submit real jobs to IBM Quantum.

It is **idempotent and resumable**:

- a completed model (`result.json`) → it is loaded and skipped;
- a chunk with saved `counts` → it is skipped;
- a previously submitted job that is still `QUEUED`/`RUNNING` → it is recovered using its `job_id`;
- if the kernel stopped after `submit` but before saving the `job_id` → the job is located using its **persistent IBM tag**;
- only a chunk with no checkpoint and no corresponding remote job is submitted as a new job.

Therefore, if the internet connection drops, the notebook is closed, or the queue takes too long, **run this same cell again**. It will resume from where it stopped.

`MAX_LOCAL_WAIT_SECONDS` only controls how long the notebook waits locally. A local timeout **does not cancel the IBM job**.


In [ ]:
# -------------------------------------------------------------------------
# First recover everything already submitted in this run.
# -------------------------------------------------------------------------
existing_jobs_df = recover_ibm_jobs_for_this_run(
    service=service,
    output_root=OUTPUT_DIR,
    backend_name=IBM_BACKEND,
)

if not existing_jobs_df.empty:
    print("Jobs already conhecidos/found for this experiment:")
    display(existing_jobs_df)
else:
    print("in the job previous found for this RUN_ID.")

# -------------------------------------------------------------------------
# Load already-completed results so the consolidated list survives
# kernel restarts as well.
# -------------------------------------------------------------------------
all_results = []
failures = []

partial_file = OUTPUT_DIR / "all_results_partial.pkl"

if partial_file.exists():
    try:
        with partial_file.open("rb") as f:
            partial = pickle.load(f)
        all_results = list(partial.get("results", []))
        failures = list(partial.get("failures", []))
        print(
            f"Checkpoint consolidado carregado: "
            f"{len(all_results)} completed result(s)."
        )
    except Exception as exc:
        print(f"[warning] Could not open {partial_file}: {exc}")

# Index of already completed models.
completed_keys = {
    (
        r["selection"]["disorder"],
        r["selection"]["ansatz"],
        int(r["selection"]["random_state"]),
    )
    for r in all_results
}

for index, qrec in enumerate(selected, start=1):

    model_key = (
        qrec["disorder"],
        qrec["model_name"],
        int(qrec["random_state"]),
    )

    print("\n" + "=" * 90)
    print(
        f"[{index}/{len(selected)}] "
        f"{qrec['disorder']} | {qrec['model_name']} "
        f"| seed={qrec['random_state']}"
    )
    print("=" * 90)

    # result consolidado already loaded.
    if model_key in completed_keys:
        print("[resume] Model already completed in the consolidated checkpoint. Skipping.")
        continue

    classical = choose_best_classical_same_split(
        qrec,
        records,
    )

    try:
        bundle = reconstruct_test_set(
            df=df,
            record=qrec,
            test_size=TEST_SIZE,
            column_main=COLUMN_MAIN,
            column_specific=COLUMN_SPECIFIC,
            group_control=GROUP_CONTROL,
            columns_exclude=COLUMNS_EXCLUDE,
        )

        result = validate_one_quantum_record_resumable(
            service=service,
            run_id=RUN_ID,
            record=qrec,
            classical_record=classical,
            dataset_bundle=bundle,
            backend=backend,
            shots=SHOTS,
            optimization_level=OPTIMIZATION_LEVEL,
            seed_transpiler=SEED_TRANSPILER,
            max_circuits_per_job=MAX_CIRCUITS_PER_JOB,
            output_root=OUTPUT_DIR,
            enable_dd=DYNAMICAL_DECOUPLING,
            gate_twirling=GATE_TWIRLING,
            measurement_twirling=MEASUREMENT_TWIRLING,
            poll_seconds=POLL_SECONDS,
            max_local_wait_seconds=MAX_LOCAL_WAIT_SECONDS,
        )

        all_results.append(result)
        completed_keys.add(model_key)

        # Atomic checkpoint after each complete model.
        atomic_pickle_dump(
            partial_file,
            {
                "run_id": RUN_ID,
                "results": all_results,
                "failures": failures,
                "updated_at_utc": utc_now(),
            },
        )

    except TimeoutError as exc:
        # IMPORTANT: a local timeout is NOT a job failure.
        # Do not mark it as a failure; only stop so the user can resume later.
        print("\n" + "!" * 90)
        print(str(exc))
        print(
            "Local execution will stop now. "
            "The IBM job still exists. Run this cell again later."
        )
        print("!" * 90)
        break

    except KeyboardInterrupt:
        print(
            "\nManual interruption detected. "
            "Previously submitted jobs remain on IBM and will be recovered "
            "when this cell is run again."
        )
        break

    except Exception as exc:
        # A real failure is recorded, but the job/chunk checkpoint is preserved.
        failure = {
            "time_utc": utc_now(),
            "run_id": RUN_ID,
            "disorder": qrec["disorder"],
            "model_name": qrec["model_name"],
            "config_key": qrec["config_key"],
            "repeat_index": qrec["repeat_index"],
            "random_state": qrec["random_state"],
            "exception": repr(exc),
            "traceback": traceback.format_exc(),
        }

        failures.append(failure)

        atomic_save_json(
            OUTPUT_DIR
            / (
                f"FAIL__{safe_name(qrec['disorder'])}"
                f"__{safe_name(qrec['model_name'])}.json"
            ),
            failure,
        )

        atomic_pickle_dump(
            partial_file,
            {
                "run_id": RUN_ID,
                "results": all_results,
                "failures": failures,
                "updated_at_utc": utc_now(),
            },
        )

        print(failure["traceback"])

# Job status when the cell finishes/pauses.
jobs_now_df = recover_ibm_jobs_for_this_run(
    service=service,
    output_root=OUTPUT_DIR,
    backend_name=IBM_BACKEND,
)

if not jobs_now_df.empty:
    print("\nEstado atual of the jobs of this campanha:")
    display(jobs_now_df)


### 9A. Only query previously submitted jobs

This cell **does not submit anything**. Use it at any time to inspect the status
of the jobs in this validation campaign after reopening the notebook or restoring the internet connection.


In [ ]:
jobs_recovered = recover_ibm_jobs_for_this_run(
    service=service,
    output_root=OUTPUT_DIR,
    backend_name=IBM_BACKEND,
)

display(jobs_recovered)

if not jobs_recovered.empty:
    print("\nResumo by status:")
    display(
        jobs_recovered.groupby("status")
        .size()
        .rename("n_jobs")
        .reset_index()
    )


## 10. IBM × simulator × classical summary


In [ ]:
summary_rows = []

for result in all_results:
    summary_rows.append({
        "disorder": result["selection"]["disorder"],
        "quantum_model": result["selection"]["ansatz"],
        "random_state": result["selection"]["random_state"],
        "backend": result["ibm"]["backend_name"],
        "shots": result["ibm"]["shots_per_circuit"],

        "f1_simulator_saved": result["comparison"]["f1_simulator_saved"],
        "f1_ibm": result["comparison"]["f1_ibm"],
        "f1_classical_saved_same_split": result["comparison"][
            "f1_classical_saved_same_split"
        ],

        "delta_ibm_minus_simulator": result["comparison"][
            "delta_ibm_minus_simulator"
        ],
        "delta_ibm_minus_classical": result["comparison"][
            "delta_ibm_minus_classical"
        ],
    })

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "ibm_vs_simulator_vs_classical.csv"
summary_df.to_csv(summary_csv, index=False)

display(summary_df)

print(f"Summary saved to: {summary_csv}")


## 11. Final QPU snapshot and consolidated file


In [ ]:
backend_info_after = backend_snapshot(backend)

save_json(
    OUTPUT_DIR / "backend_snapshot_after.json",
    backend_info_after,
)

final_manifest = dict(run_manifest)
final_manifest.update({
    "finished_at_utc": utc_now(),
    "backend_snapshot_after": backend_info_after,
    "n_success": len(all_results),
    "n_failures": len(failures),
    "failures": failures,
})

save_json(
    OUTPUT_DIR / "run_manifest_final.json",
    final_manifest,
)

with (OUTPUT_DIR / "all_results.pkl").open("wb") as f:
    pickle.dump(
        {
            "manifest": final_manifest,
            "results": all_results,
            "failures": failures,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print("Validation completed.")
print(f"Successes: {len(all_results)}")
print(f"Failures: {len(failures)}")
print(f"Directory: {OUTPUT_DIR.resolve()}")


## 12. Inspect details of an experiment

Example: IBM jobs, transpiled depth, number of gates, probabilities, counts, and metrics.


In [ ]:
# example:
if all_results:
    r = all_results[0]

    print("disorder:", r["selection"]["disorder"])
    print("Ansatz:", r["selection"]["ansatz"])
    print("Backend:", r["ibm"]["backend_name"])

    print("\nIBM metrics:")
    display(pd.DataFrame([r["ibm"]["metrics"]]).drop(
        columns=["confusion_matrix", "classification_report"],
        errors="ignore",
    ))

    print("\nJobs:")
    jobs = r["ibm"]["jobs"]
    if jobs:
        display(pd.DataFrame(jobs))

    print("\nTranspilation of the primeiro estimador:")
    display(pd.DataFrame([
        r["ibm"]["estimators"][0]["transpilation"]["original"],
        r["ibm"]["estimators"][0]["transpilation"]["isa"],
    ], index=["logical", "ISA"]))
